# LLMs for Low-Resource Languages

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 1 · Monday 15 June 2026

**Lecturer:** Robiert Sepúlveda Torres and Iván Martínez Murrillo
> Before running anything, make sure the kernel is **NLPAICS 02** (menu: *Kernel → Change Kernel*). It should already be selected.


This notebook demonstrates end-to-end LLM-based translation evaluation:
1. A small multilingual model generates translations.
2. The Selene judge model (Atla AI) scores them with a rubric.
3. You score the same translations manually.
4. We compare automatic and human scores.

**All you need to adapt this notebook is to change the variables in Section 0.**


## 0 · Environment check

In [ ]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 02"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")


## Section 1: Configuration

In [ ]:
# ============================================================
# CONFIGURATION — edit only this cell to adapt the notebook
# ============================================================

# Model to use for translation (must support the target language).
# Alternatives: "google/gemma-2-2b-it", "meta-llama/Llama-3.2-3B-Instruct" (requires HF_TOKEN)
# Note: Qwen3 models ("Qwen/Qwen3-3B-Instruct") require accepting a license on HuggingFace
#       (hf auth login + accept terms at huggingface.co/Qwen/Qwen3-3B-Instruct) before they
#       can be downloaded. Qwen2.5 below is identical in size and fully public.
import os
os.environ["HF_HUB_CACHE"] = "/workspace/.hf_home/hub"

GENERATOR_MODEL_ID: str = "Qwen/Qwen2.5-3B-Instruct"

# Target language for translation (any language the model supports).
TARGET_LANGUAGE: str = "Valencian"  # <-- change this to your target language

# Four English sentences to translate and evaluate.
# Edit these to reflect the linguistic phenomena you want to explore.
SENTENCES: list[str] = [
    "To be wiped out.",
    "A colourful school of fish swam gracefully through the coral reef.",
    "To call the shots.",
    "The festival celebrates the harvest season with traditional music and dance.",
]

# Your reference (gold standard) translations into TARGET_LANGUAGE.
# Write one correct translation per sentence, in the same order as SENTENCES.
# These are displayed alongside the model output in Section 5 to help you score,
# and appear in the comparison table in Section 6.
GOLD_STANDARDS: list[str] = [
    "Estar fet una coca.",  # <-- write your translation for sentence 1
    "Un acolorit banc de peixos nadava amb gràcia a través de l'escull de coral.",  # <-- write your translation for sentence 2
    "Ser qui mana.", # <-- write your translation for sentence 3
    "El festival celebra la temporada de la collita amb música i dansa tradicionals.", # <-- write your translation for sentence 4
]

# HuggingFace access token (required only for gated models such as Llama 3.2 or Qwen3).
# Leave as None when using Qwen2.5 or Gemma.
HF_TOKEN: str | None = None

## Section 2: Setup and Model Loading

We load the generator model using `AutoModelForCausalLM` and `AutoTokenizer` from `transformers`.
Using `device_map="auto"` and `torch_dtype=torch.bfloat16` reduces VRAM usage.

> **VRAM note**: a 3B model in bfloat16 typically uses around 8GB of VRAM.

In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

from notebook_helpers import (
    build_translation_prompt,
    build_judge_prompt,
    parse_judge_response,
    validate_human_scores,
)

# Load the generator tokenizer and model.
print(f"Loading generator model: {GENERATOR_MODEL_ID}")
gen_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_ID,
    token=HF_TOKEN,
)
gen_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
)
gen_model.eval()
print("Generator model loaded.")

## Section 3: Evaluation Rubric and Sentences

The rubric below defines the three evaluation criteria and the meaning of each score (1–5).
The same rubric is used by both the automatic judge and you (the human evaluator) in Sections 4 and 5.

In [ ]:
# Three-criterion translation evaluation rubric.
# Each criterion is scored 1–5; the description below defines what each score means.
RUBRIC: dict[str, str] = {
    "score_adequacy": (
        "Adequacy / fidelity: Does the translation convey the full meaning of the source sentence without omissions, additions, or distortions?\n"
        "Score 1: Major meaning loss: the translation omits or severely distorts the core proposition; a reader cannot recover the source meaning.\n"
        "Score 2: Significant gaps: the general topic is conveyed, but one or more important elements (arguments, negation, key modifiers) are missing or changed.\n"
        "Score 3: Acceptable: the main meaning is preserved with minor imprecisions that do not mislead the reader.\n"
        "Score 4: Good: all meaning elements are present; at most one small imprecision that does not affect comprehension.\n"
        "Score 5: Complete and accurate: the translation fully and faithfully conveys every element of the source sentence."
    ),
    "score_fluency": (
        "Fluency / naturalness: Does the translation read naturally in the target language, as a native speaker would write it?\n"
        "Score 1: Unreadable: so unnatural that it is difficult to process as the target language.\n"
        "Score 2: Disfluent: understandable but clearly non-native; noticeable awkward phrasing throughout.\n"
        "Score 3: Acceptable: mostly readable with a few unnatural phrasings that do not impede comprehension.\n"
        "Score 4: Good: reads naturally for most of the sentence; one minor phrasing a native speaker would likely improve.\n"
        "Score 5: Fully natural: reads exactly as a fluent native speaker would write it."
    ),
    "score_grammar": (
        "Grammatical correctness: Are morphological forms, agreement, and syntax correct in the target language?\n"
        "Score 1: Pervasive errors: agreement, morphology, or syntax errors in nearly every clause; the text is barely parseable.\n"
        "Score 2: Frequent errors: multiple clear grammatical errors that affect readability.\n"
        "Score 3: Minor errors: one or two grammatical errors that do not impede comprehension.\n"
        "Score 4: Mostly correct: at most one minor grammatical issue a careful reader would notice.\n"
        "Score 5: Grammatically perfect: no morphological, agreement, or syntactic errors."
    ),
}

print("Sentences to evaluate:")
for i, s in enumerate(SENTENCES, 1):
    print(f"  {i}. {s}")
print(f"\nTarget language: {TARGET_LANGUAGE}")
print("\nEvaluation rubric:")
for criterion, description in RUBRIC.items():
    print(f"\n{criterion}:")
    for line in description.split("\n"):
        print(f"  {line}")

## Section 4: Translation Generation

We iterate over the four sentences, build a translation prompt for each, and run inference
with the generator model. Because the generator is an instruction-tuned chat model, we wrap
the helper-built prompt in `apply_chat_template` before tokenisation. Results are stored
in a pandas DataFrame.

In [ ]:
def run_translation(source: str) -> str:
    """Generate a translation of source into TARGET_LANGUAGE using the generator model."""
    user_message = build_translation_prompt(source, TARGET_LANGUAGE)
    messages = [{"role": "user", "content": user_message}]
    tokens = gen_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    # transformers 5.x returns BatchEncoding; 4.x returned a plain tensor.
    if isinstance(tokens, torch.Tensor):
        tokens = {"input_ids": tokens}
    tokens = {k: v.to(gen_model.device) for k, v in tokens.items()}
    with torch.no_grad():
        output_ids = gen_model.generate(
            **tokens,
            max_new_tokens=128,
            do_sample=False,
        )
    # Decode only the newly generated tokens (skip the prompt).
    new_tokens = output_ids[0][tokens["input_ids"].shape[1]:]
    translation = gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    if not translation:
        raise ValueError(f"Generator returned an empty translation for: {source!r}")
    return translation


results: list[dict[str, str]] = []
for sentence in SENTENCES:
    translation = run_translation(sentence)
    results.append({"source": sentence, "translation": translation})
    print(f"Source:      {sentence}")
    print(f"Translation: {translation}")
    print()

df = pd.DataFrame(results)
print("Translation DataFrame:")
df

## Section 5: Automatic Evaluation (Selene Judge)

We now unload the generator model to free VRAM, then load the Selene judge model.
Selene receives each source sentence, its translation, and the rubric, and returns
a JSON object with per-criterion scores (1–5) and a justification string.
Selene is also an instruction-tuned chat model (Llama-3.1-8B base), so we use
`apply_chat_template` with a system + user message pair.

> **VRAM note**: Selene (8B) in fp16 requires approximately 16 GB of VRAM.

In [ ]:
# Unload the generator model to free VRAM before loading the judge.
del gen_model
del gen_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Generator model unloaded.")

JUDGE_MODEL_ID = "AtlaAI/Selene-1-Mini-Llama-3.1-8B"
print(f"Loading judge model: {JUDGE_MODEL_ID}")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
judge_model.eval()
print("Judge model loaded.")

In [ ]:
def run_judge(source: str, translation: str) -> dict[str, int | str]:
    """Run Selene on a (source, translation) pair and return parsed scores."""
    user_message = build_judge_prompt(source, translation, RUBRIC)
    messages = [
        {"role": "system", "content": "You are an expert translation evaluator."},
        {"role": "user", "content": user_message},
    ]
    tokens = judge_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    # transformers 5.x returns BatchEncoding; 4.x returned a plain tensor.
    if isinstance(tokens, torch.Tensor):
        tokens = {"input_ids": tokens}
    tokens = {k: v.to(judge_model.device) for k, v in tokens.items()}
    with torch.no_grad():
        output_ids = judge_model.generate(
            **tokens,
            max_new_tokens=512,
            do_sample=False,
        )
    new_tokens = output_ids[0][tokens["input_ids"].shape[1]:]
    raw = judge_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    try:
        return parse_judge_response(raw)
    except ValueError as exc:
        print(f"[WARNING] Could not parse judge output. Raw:\n{raw}\nError: {exc}")
        print("Tip: try increasing max_new_tokens or re-running this cell.")
        raise


judge_rows: list[dict[str, int | str]] = []
for _, row in df.iterrows():
    scores = run_judge(str(row["source"]), str(row["translation"]))
    judge_rows.append(scores)
    print(f"Source: {row['source']}")
    print(f"Judge scores: {scores}")
    print()

judge_df = pd.DataFrame(judge_rows)
df = pd.concat([df, judge_df], axis=1)
print("DataFrame with judge scores:")
df

## Section 6: Human Evaluation

Now it is your turn. For each sentence and its translation, assign a score from 1 to 5
for each rubric criterion. Edit the `human_scores` list below — **replace every 0 with
your actual score before running the next section**.

Refer to the rubric defined in **Section 3** for the exact meaning of each score value.
As a quick reference:

| Score | Adequacy | Fluency | Grammar |
|-------|----------|---------|---------|
| **1** | Major meaning loss | Unreadable | Pervasive errors |
| **2** | Significant gaps | Disfluent | Frequent errors |
| **3** | Acceptable | Acceptable | Minor errors |
| **4** | Good | Good | Mostly correct |
| **5** | Complete and accurate | Fully natural | Grammatically perfect |

In [ ]:
print("Sentences to score (source | gold standard | model translation):")
for i, row in df[["source", "translation"]].iterrows():
    idx = int(i)
    print(f"\n  Sentence {idx + 1}:")
    print(f"    EN (source)       : {row['source']}")
    print(f"    {TARGET_LANGUAGE} (gold standard) : {GOLD_STANDARDS[idx]}")
    print(f"    {TARGET_LANGUAGE} (model output)  : {row['translation']}")
print()

# Fill in your scores here. Replace 0 with a number from 1 to 5.
# One dict per sentence, in the same order as SENTENCES.
# See the rubric in Section 2 for the exact meaning of each score.
human_scores: list[dict[str, int]] = [
    # Sentence 1
    {
        "score_adequacy": 0,  # <-- replace with your score (1-5)
        "score_fluency": 0,   # <-- replace with your score (1-5)
        "score_grammar": 0,   # <-- replace with your score (1-5)
    },
    # Sentence 2
    {
        "score_adequacy": 0,
        "score_fluency": 0,
        "score_grammar": 0,
    },
    # Sentence 3
    {
        "score_adequacy": 0,
        "score_fluency": 0,
        "score_grammar": 0,
    },
    # Sentence 4
    {
        "score_adequacy": 0,
        "score_fluency": 0,
        "score_grammar": 0,
    },
]

# Validate scores before proceeding (raises ValueError if any score is 0 or out of range).
for i, scores in enumerate(human_scores):
    try:
        validate_human_scores(scores)
    except ValueError as exc:
        raise ValueError(
            f"Invalid scores for sentence {i + 1}: {exc}. "
            "Edit the human_scores list above and re-run this cell."
        ) from exc

print("Human scores validated. Proceed to Section 7.")

## Section 7: Judge vs. Human Comparison

We compute per-criterion absolute differences, mean absolute difference (MAD),
and Krippendorff's Alpha (ordinal) as an inter-rater agreement metric.

**Discussion prompts:**
- Which criterion shows the largest disagreement? Why might that be?
- Does the judge tend to score higher or lower than you?
- What does this tell us about automatic vs. human evaluation of low-resource languages?

In [ ]:
SCORE_COLS = ["score_adequacy", "score_fluency", "score_grammar"]

human_df = pd.DataFrame(human_scores)
human_df.columns = [f"human_{c}" for c in human_df.columns]

comparison = pd.concat([df.reset_index(drop=True), human_df.reset_index(drop=True)], axis=1)
comparison["gold_standard"] = GOLD_STANDARDS

for col in SCORE_COLS:
    comparison[f"diff_{col}"] = (
        comparison[col].astype(int) - comparison[f"human_{col}"].astype(int)
    ).abs()

diff_cols = [f"diff_{c}" for c in SCORE_COLS]
comparison["MAD"] = comparison[diff_cols].mean(axis=1)
comparison["percent_agreement"] = comparison[diff_cols].le(1).mean(axis=1) * 100

display_cols = (
    ["source", "gold_standard", "translation"]
    + SCORE_COLS
    + [f"human_{c}" for c in SCORE_COLS]
    + diff_cols
    + ["MAD", "percent_agreement"]
)

print("Judge vs. Human Comparison:")
print(comparison[display_cols].to_string(index=False))
print(f"\nOverall MAD:               {comparison['MAD'].mean():.2f}")
print(f"Overall percent agreement: {comparison['percent_agreement'].mean():.1f}%")
comparison[display_cols]

In [ ]:
import numpy as np
import krippendorff

# Build reliability matrices: shape (2 raters, n_items).
# Overall: flatten all criteria and all sentences into a single vector.
judge_flat = comparison[SCORE_COLS].values.flatten().astype(float)
human_flat = comparison[[f"human_{c}" for c in SCORE_COLS]].values.flatten().astype(float)

alpha_overall = krippendorff.alpha(
    reliability_data=np.array([judge_flat, human_flat]),
    level_of_measurement="ordinal",
)

print("Krippendorff's Alpha (ordinal scale):")
print(f"  Overall (all criteria, all sentences): α = {alpha_overall:.3f}")
for col in SCORE_COLS:
    j = comparison[col].astype(float).values
    h = comparison[f"human_{col}"].astype(float).values
    alpha = krippendorff.alpha(
        reliability_data=np.array([j, h]),
        level_of_measurement="ordinal",
    )
    print(f"  {col}: α = {alpha:.3f}")

print(
    f"\nNote: with only {len(SENTENCES)} sentences these statistics have high variance. "
    "Treat them as illustrative rather than definitive."
)